# Movie-Grounded Creative Director V3 — T4 Validation Notebook

This notebook runs the full director pipeline on the validated movie
using real Qwen on Tesla T4. It reproduces the exact validation flow
from `scripts/run_director_validation.py` but in an interactive format
for inspection and debugging.

**Prerequisites:**
- GPU with CUDA (T4 16GB recommended)
- Qwen/Qwen3-4B-Instruct-2507 model
- Project dependencies installed

**Expected outputs:**
- `reports/director_validation.json` — structured validation record
- `reports/director_reasoning.md` — human-readable reasoning report

In [ ]:
# Environment setup
import sys
import os
sys.path.insert(0, '/content/automovies/src')

# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load movie index
import json
from pathlib import Path

PROJECT_DIR = Path('/content/automovies/data/bc6384be-47a5-4ee8-8674-7ff861472026')
MOVIE_INDEX = PROJECT_DIR / 'movie_index.json'

with open(MOVIE_INDEX, 'r', encoding='utf-8') as f:
    movie_index = json.load(f)

print(f"Movie: {movie_index['movie']['title']}")
print(f"Duration: {movie_index['movie']['duration_sec']}s")
print(f"Scenes: {len(movie_index['scenes'])}")
for s in movie_index['scenes'][:3]:
    print(f"  {s['scene_id']}: {s['story']['location']} ({len(s['story']['objects'])} objs, {len(s['story']['actions'])} acts)")
print("  ...")

In [ ]:
# Build SceneFacts
from director.scene_facts import SceneFacts

facts = SceneFacts.from_movie_intelligence(movie_index=movie_index)
print(f"Known characters: {facts.known_characters()}")
print(f"Known locations: {facts.known_locations()}")
print(f"Known objects: {facts.known_objects()[:10]}...")
print(f"Known actions: {facts.known_actions()}")

In [ ]:
# Test context builder (two-level context)
from director.context_builder import DirectorContextBuilder

cb = DirectorContextBuilder(max_tokens=4096, reserve_for_output=2048)
ctx, meta = cb.build_concept_generation_context(
    movie_index['movie'], facts
)
print("=== CONTEXT METADATA ===")
for k, v in meta.items():
    print(f"  {k}: {v}")

# Verify inventory contains all scenes
inventory_lines = ctx.split('## FULL MOVIE INVENTORY')[1].split('## WHAT ACTUALLY EXISTS')[0].strip().split('\n')
print(f"\nInventory lines (all scenes): {len(inventory_lines)}")
print(f"First: {inventory_lines[0]}")
print(f"Last:  {inventory_lines[-1]}")

In [ ]:
# Initialize Qwen provider
from director.providers.qwen import QwenProvider

provider = QwenProvider(
    model_id="Qwen/Qwen3-4B-Instruct-2507",
    dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.9,
)

# Warm-up call
print("Loading model...")
_ = provider.generate_text("Say hello in one word.")
print("Model loaded.")

In [ ]:
# Run full director pipeline
from director.grounded import MovieGroundedDirector
from director.memory import CreativeMemory

memory_dir = PROJECT_DIR / 'memory'
memory_dir.mkdir(exist_ok=True)

director = MovieGroundedDirector(
    llm=provider.generate_text,
    memory_dir=memory_dir,
    context_tokens=4096,
    reserve_for_output=2048,
)

result = director.develop(
    movie_metadata=movie_index['movie'],
    scale_facts=facts,
    num_concepts=6,
    min_coverage=0.4,
    duration_sec=90,
)

print(f"Verdict: {result.get('verdict', 'UNKNOWN')}")
print(f"Selected concept: {result.get('selected_concept', {}).get('title', 'NONE')}")
print(f"Plan: {'YES' if result.get('plan') else 'NO'}")
print(f"LLM calls: {result.get('llm_stats', {}).get('llm_calls')}")
print(f"Regeneration rounds: {result.get('llm_stats', {}).get('regeneration_rounds')}")

# Print selected concept details
sel = result.get('selected_concept')
if sel:
    print(f"\nSelected thesis: {sel.get('thesis')}")
    ev = sel.get('_evidence')
    if ev:
        print(f"Reference coverage: {ev.get('reference_coverage')}")
        print(f"Claim coverage: {ev.get('claim_coverage_detail')}")
        print(f"Claims: {len(ev.get('claims', []))}")
        for c in ev.get('claims', []):
            print(f"  [{c['claim_id']}] {c['type']} → {c['status']} (conf={c['confidence']})" )

# Plan audit
plan = result.get('plan')
if plan:
    audit = plan.get('grounding_audit', {})
    print(f"\nPlan audit:")
    print(f"  Coverage: {audit.get('coverage')}")
    print(f"  Sufficient: {audit.get('sufficient')}")
    print(f"  Invented: {audit.get('invented_terms')}")

In [ ]:
# Save validation report
from director.report import write_report
from director.evidence import EvidenceAnalyzer

analyzer = EvidenceAnalyzer(facts)
report_path = write_report(
    project_dir=PROJECT_DIR,
    result=result,
    analyzer=analyzer,
)
print(f"Report written to: {report_path}")

In [ ]:
# Display reasoning report
md_path = PROJECT_DIR / 'reports' / 'director_reasoning.md'
if md_path.exists():
    with open(md_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # Print first 5000 chars
    print(content[:5000])
    print("...\n[truncated]")
else:
    print("Reasoning report not found")

In [ ]:
# Validation summary for milestone tracking
print("=== V3 VALIDATION SUMMARY ===")
print(f"Movie: {movie_index['movie']['title']}")
print(f"Total scenes: {len(movie_index['scenes'])}")
print(f"Verdict: {result.get('verdict')}")
print(f"Concepts generated: {len(result.get('generated_concepts', []))}")
print(f"Rejected: {len(result.get('rejected_concepts', []))}")
sel = result.get('selected_concept')
if sel:
    print(f"Selected: {sel.get('title')}")
    ev = sel.get('_evidence')
    if ev:
        print(f"  Reference coverage: {ev.get('reference_coverage')}")
        print(f"  Claim coverage: {ev.get('claim_coverage_detail')}")
        print(f"  Claims: {len(ev.get('claims', []))}")
print(f"Plan status: {'PASS' if result.get('plan') else 'REJECTED/FAIL'}")
if result.get('plan_rejection'):
    print(f"Plan rejection: {result['plan_rejection'].get('reason')}")
print(f"LLM calls: {result.get('llm_stats', {}).get('llm_calls')}")
print(f"Regeneration rounds: {result.get('llm_stats', {}).get('regeneration_rounds')}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Model: Qwen/Qwen3-4B-Instruct-2507 (4-bit)")